# 02 — Factor Research

Compute the **Alpha158** feature set (qlib, from scratch) plus classical factors, then evaluate predictive power with an **alphalens-style** IC / ICIR / turnover / quantile-returns report and a factor-decay analysis.

In [1]:
import sys, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Put the quantcortex repo root on the path (notebooks live in research/).
for p in ("..", "."):
    ap = os.path.abspath(p)
    if ap not in sys.path:
        sys.path.insert(0, ap)

RNG = np.random.default_rng(42)

def load_prices(symbols, start="2018-01-01", periods=1800):
    """Real prices via yfinance if available, else deterministic synthetic GBM."""
    try:
        from data.providers.yfinance_provider import YFinanceProvider
        px = YFinanceProvider().get_prices(list(symbols), start=start)
        if px is not None and not px.empty and px.shape[0] > 120:
            return px.dropna(how="all").ffill().dropna()
        raise RuntimeError("empty frame")
    except Exception as exc:
        print(f"[offline] yfinance unavailable ({type(exc).__name__}); using synthetic data.")
    dates = pd.bdate_range(start, periods=periods)
    mu = RNG.normal(0.0003, 0.00015, len(symbols))
    vol = RNG.uniform(0.008, 0.018, len(symbols))
    shocks = RNG.normal(mu, vol, size=(periods, len(symbols)))
    return pd.DataFrame(100 * np.exp(np.cumsum(shocks, axis=0)),
                        index=dates, columns=list(symbols))

def synth_ohlcv(close):
    """Build an OHLCV frame around a close series (synthetic intrabar range)."""
    close = close.dropna()
    hi = close * (1 + np.abs(RNG.normal(0, 0.004, len(close))))
    lo = close * (1 - np.abs(RNG.normal(0, 0.004, len(close))))
    op = close.shift(1).fillna(close.iloc[0])
    vol = RNG.integers(1_000_000, 6_000_000, len(close)).astype(float)
    return pd.DataFrame({"open": op, "high": hi, "low": lo, "close": close, "volume": vol})

print("quantcortex research environment ready.")


quantcortex research environment ready.


In [2]:
symbols = ["AAPL", "MSFT", "AMZN", "NVDA", "JPM", "XOM", "PG", "KO"]
prices = load_prices(symbols)
returns = prices.pct_change().dropna()
print("universe:", list(prices.columns), prices.shape)

[offline] yfinance unavailable (ImportError); using synthetic data.
universe: ['AAPL', 'MSFT', 'AMZN', 'NVDA', 'JPM', 'XOM', 'PG', 'KO'] (1800, 8)


## Alpha158 features (single symbol)

In [3]:
from alpha.feature_engineering.alpha158 import Alpha158

ohlcv = synth_ohlcv(prices[symbols[0]])
feats = Alpha158().compute(ohlcv)
print("Alpha158 produced", feats.shape[1], "features")
assert feats.shape[1] == 158
feats.dropna().iloc[-1].head(12)

Alpha158 produced 158 features


KMID     0.000675
KLEN     0.006828
KMID2    0.098919
KUP      0.001726
KUP2     0.252797
KLOW     0.004427
KLOW2    0.648284
KSFT     0.003376
KSFT2    0.494407
OPEN0    0.999325
HIGH0    1.001725
LOW0     0.994901
Name: 2024-11-22 00:00:00, dtype: float64

## Classical cross-sectional factors

In [4]:
from alpha.factors.classical.momentum import MomentumFactor
from alpha.factors.classical.low_vol import LowVolFactor

mom = MomentumFactor(lookback=126, gap=21)
mom_panel = mom.compute(prices)
mom_z = mom.cross_sectional_zscore(mom_panel)
lowvol_panel = LowVolFactor(window=63).compute(prices)
print("momentum panel:", mom_panel.shape, "| non-null rows:", int(mom_panel.notna().any(axis=1).sum()))
mom_z.dropna(how="all").tail(3)

momentum panel: (1800, 8) | non-null rows: 1674


,AAPL,MSFT,AMZN,NVDA,JPM,XOM,PG,KO
2024-11-20,-0.593235,1.001530,1.688514,-1.193646,-1.273864,0.803982,-0.272119,-0.161162
2024-11-21,-0.615450,1.023761,1.686348,-1.176331,-1.308684,0.752977,-0.170563,-0.192058
2024-11-22,-0.578762,1.209300,1.491691,-1.192987,-1.301178,0.861105,-0.250095,-0.239074


## Alphalens report (IC, ICIR, turnover, quantiles)

In [5]:
from alpha.validation.alphalens_report import AlphalensReport

fwd_returns = prices.pct_change(5).shift(-5)   # 5-day forward return
factor = mom_z.dropna(how="all")
report = AlphalensReport(factor, fwd_returns, quantiles=4).compute()
turn = float(np.asarray(report["turnover"]).mean())
ls = float(np.asarray(report["long_short_return"]).mean())
print("IC mean  : %+.4f" % report["ic_mean"])
print("ICIR     : %+.4f" % report["icir"])
print("IC t-stat: %+.3f" % report["ic_tstat"])
print("turnover : %.3f" % turn)
print("long-short: %+.5f" % ls)

IC mean  : -0.0138
ICIR     : -0.0366
IC t-stat: -1.496
turnover : nan
long-short: -0.00088


## Factor decay

In [6]:
from alpha.validation.factor_decay import FactorDecay

decay = FactorDecay().compute(factor, returns, max_lag=10)
print(decay.round(4).to_string())

     ic_mean  ic_std    icir
lag                         
0    -0.0135  0.3802 -0.0355
1    -0.0095  0.3807 -0.0249
2    -0.0057  0.3841 -0.0149
3    -0.0089  0.3815 -0.0233
4    -0.0083  0.3807 -0.0217
5    -0.0138  0.3774 -0.0366
6    -0.0102  0.3749 -0.0273
7    -0.0163  0.3733 -0.0437
8    -0.0167  0.3705 -0.0451
9    -0.0206  0.3686 -0.0560
10   -0.0225  0.3742 -0.0600


In [7]:
ic = report["ic"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ic.cumsum().plot(ax=ax[0]); ax[0].set_title("Cumulative IC"); ax[0].axhline(0, color="k", lw=0.5)
decay["ic_mean"].plot(kind="bar", ax=ax[1]); ax[1].set_title("Mean IC by forward lag")
plt.tight_layout(); print("factor research complete.")

factor research complete.
